# Подготовка датасета на 10к объектов

In [ ]:
# ! wget https://archive.ics.uci.edu/static/public/280/higgs.zip
# !unzip /content/higgs.zip
# !gunzip /content/HIGGS.csv.gz
# import pandas as pd
# df = pd.read_csv('/content/HIGGS.csv', header=None)

from sklearn.model_selection import train_test_split
X_train, X_test = train_test_split(df, test_size=0.00091, random_state=42, stratify=df[0])
column_names = ['class_label', 'jet_1_b-tag', 'jet_1_eta', 'jet_1_phi', 'jet_1_pt', 'jet_2_b-tag', 'jet_2_eta', 'jet_2_phi', 'jet_2_pt', 'jet_3_b-tag', 'jet_3_eta', 'jet_3_phi', 'jet_3_pt', 'jet_4_b-tag', 'jet_4_eta', 'jet_4_phi', 'jet_4_pt', 'lepton_eta', 'lepton_pT', 'lepton_phi', 'm_bb', 'm_jj', 'm_jjj', 'm_jlv', 'm_lv', 'm_wbb', 'm_wwbb', 'missing_energy_magnitude', 'missing_energy_phi']
result = X_test.head(10000)
result.columns = column_names
result.to_csv('HIGGS_10k_sample.csv', index=False)

--2025-03-26 09:15:37--  https://archive.ics.uci.edu/static/public/280/higgs.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘higgs.zip.1’

higgs.zip.1             [         <=>        ] 171.68M  33.4MB/s               ^C
Archive:  /content/higgs.zip
  inflating: HIGGS.csv.gz            gzip: /content/HIGGS.csv already exists; do you wish to overwrite (y or n)? ^C


# Само задание

В этом блокноте мы рассмотрим применение методов машинного обучения для решения задачи из области физики, а именно для поиска экзотических частиц в данных, полученных на коллайдерах высоких энергий.

Задача состоит в классификации смоделированных событий столкновений частиц на "сигнал" (события с каскадом распадов, включающим бозоны Хиггса) и "фон" (другие процессы, например, рождение пары топ-кварков).

Для решения этой задачи используются 28 признаков, характеризующих продукты распада частиц. Эти признаки делятся на две группы:


*   первый 21 признак - это "low-level features", кинематические свойства частиц (например, импульс и углы), измеренные непосредственно детекторами.
*   оставшиеся 7 признаков - это "high-level features", вычисленные физиками на основе low-level признаков и физических принципов (например, инвариантные массы реконструированных W-бозонов и бозонов Хиггса) для улучшения разделения сигнала и фона.


Цель применения методов машинного обучения в данном контексте - автоматизировать процесс создания признаков, потенциально устранив необходимость в ручной разработке high-level features физиками. Подробнее про саму задачу и получение High-level признаков можно прочитать [в оригинальной статье](https://arxiv.org/pdf/1402.4735).

[Исходный датасет](https://archive.ics.uci.edu/dataset/280/higgs) содержит 11 миллионов строк. Очевидно, что обработка такого набора данных достаточно трудоемкая процедура, требующая весьма внушительных ресурсов. В связи с этим


1.   Не будем задаваться целью повторить результаты, описанные в оригинальной статье, ограничившись лишь некоторыми скромными изысканиями.
2.   Рассмотрим лишь небольшой фрагмент данных, содержащий 10 тысяч строк.



## Загузка данных

In [1]:
!wget https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/SMILE_data/HIGGS_10k_sample.csv

import pandas as pd

df = pd.read_csv('/content/HIGGS_10k_sample.csv')
df.head()

--2025-03-27 09:43:44--  https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/SMILE_data/HIGGS_10k_sample.csv
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5061771 (4.8M) [text/csv]
Saving to: ‘HIGGS_10k_sample.csv’

HIGGS_10k_sample.cs 100%[===================>]   4.83M  4.66MB/s    in 1.0s    

2025-03-27 09:43:46 (4.66 MB/s) - ‘HIGGS_10k_sample.csv’ saved [5061771/5061771]



,class_label,jet_1_b-tag,jet_1_eta,jet_1_phi,jet_1_pt,jet_2_b-tag,jet_2_eta,jet_2_phi,jet_2_pt,jet_3_b-tag,...,lepton_phi,m_bb,m_jj,m_jjj,m_jlv,m_lv,m_wbb,m_wwbb,missing_energy_magnitude,missing_energy_phi
0,1.0,0.404816,1.469665,-1.294180,0.372943,-0.555710,1.199138,0.192079,1.618937,2.173076,...,1.141340,-0.600127,0.000000,1.106207,1.129289,0.977360,0.888801,1.235912,1.200951,0.992985
1,1.0,0.998681,-1.798000,-0.654980,0.695750,-1.320247,0.412782,-1.659651,-1.710747,0.000000,...,0.156943,0.477618,3.101961,0.892487,0.958872,0.979686,0.584742,0.605006,0.714586,0.654367
2,0.0,0.639068,-0.596123,-0.265465,0.956584,0.880806,0.776005,-0.812014,-0.767200,2.173076,...,-1.051484,1.488113,3.101961,2.402442,1.587722,0.989521,0.947525,1.255901,1.184860,0.994325
3,1.0,0.500164,1.322595,1.084061,1.228268,-0.241178,1.357527,0.302985,1.598425,2.173076,...,0.631652,0.750635,3.101961,0.770955,0.695171,1.009457,1.351649,0.754952,0.894234,0.867700
4,0.0,0.582518,0.091499,-0.115097,1.058710,1.091686,0.625677,1.071403,0.473043,0.000000,...,-1.339640,-1.707176,0.000000,0.789051,0.918229,0.987382,0.851245,1.071376,0.962279,0.845701


Разобьем датасет на 2 части

*   **Low-level** (21 колонока) -- признаки, отвечающие кинематическими свойствами, измеренными детекторами частиц в ускорителе
*   **High-level** (7 колонок) -- признаки высокого уровня, полученные физиками на основе 21 низкоуровневого признака


In [2]:
X_low = df.iloc[:,1:22]
X_high = df.iloc[:,22:]
y = df.iloc[:,0]

## Первое приближение

Сначала рассмотрим Low-level данные. В качестве модели будем использовать логистическую регрессию. Разобьем данные на тренировочную и тестовую части для, соответственно, обучения и оценки модели.

In [150]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

random_state = 42

X_train_low, X_test_low, y_train, y_test = train_test_split(X_low, y, test_size=0.2, random_state=random_state)
clf = LogisticRegression().fit(X_train_low, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_low)[:, 1]))

0.5910361276392759


Не самые хорошие результаты. Рассмотрим теперь только High-level данные и выполним аналогичные действия

In [151]:
X_train_high, X_test_high, y_train, y_test = train_test_split(X_high, y, test_size=0.2, random_state=random_state)
clf = LogisticRegression().fit(X_train_high, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_high)[:, 1]))

0.6242615048580251


Значение метрики увеличилось, очевидно, что предложенный способ генерации признаков имеет смысл. Проверим, какого качества удастся добиться на комбинации Low-level и High-level признаков (Complete).

In [152]:
X_train_complete, X_test_complete, y_train, y_test = train_test_split(df.iloc[:,1:], y, test_size=0.2, random_state=random_state)
clf = LogisticRegression().fit(X_train_complete, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_complete)[:, 1]))

0.6705832140794858


Качество модели еще значительно выросло

## Полиномиальные признаки

Попробуем автоматизировать работу физиков и сгенерировать дополнительные признаки, используя `PolynomialFeatures` для набора Low-level.

In [153]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(2)

X_train_low_poly = poly.fit_transform(X_train_low)
X_test_low_poly = poly.transform(X_test_low)

# Признаков стало заметно больше. Для сходимости оптимизатора увеличим число итераций в модели.
clf = LogisticRegression(max_iter=1000).fit(X_train_low_poly, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_low_poly)[:, 1]))

0.6413948934247193


Как видно, результат для Low-level с использованием `PolynomialFeatures` оказался лучше, чем просто использование High-level. Как будто у нас получилось добиться некоторой автоматизации процесса генерации признаков.

На самом деле причина в простоте модели логистической регрессии, которая не в состоянии выявить сложные зависимости в исходных данных. В таком случае `PolynomialFeatures` улучшает результаты и для High-level, и для Complete.

In [154]:
poly = PolynomialFeatures(2)

X_train_high_poly = poly.fit_transform(X_train_high)
X_test_high_poly = poly.transform(X_test_high)

clf = LogisticRegression(max_iter=1000).fit(X_train_high_poly, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_high_poly)[:, 1]))

0.6822715974994351


In [155]:
poly = PolynomialFeatures(2)

X_train_complete_poly = poly.fit_transform(X_train_complete)
X_test_complete_poly = poly.transform(X_test_complete)

clf = LogisticRegression(max_iter=1000).fit(X_train_complete_poly, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_complete_poly)[:, 1]))

0.6859159950792096


## Ансамблирование

Попробуем показать, что рассматриваемые признаки действительно не так просты как нам кажется. Попробуем применить ансамбли моделей (в нашем случае `RandomForest`) для выявления более сложных зависимостей.

Сначала для Low-level.

In [156]:
from sklearn.ensemble import RandomForestClassifier

# Используем потенциально достаточно глубокие деревья
clf = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=random_state).fit(X_train_low, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_low)[:, 1]))

0.6559503903994377


High-level

In [157]:
clf = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=random_state).fit(X_train_high, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_high)[:, 1]))

0.754060907333484


И complete

In [158]:
clf = RandomForestClassifier(n_estimators=300, max_depth=15, random_state=random_state).fit(X_train_complete, y_train)
print(roc_auc_score(y_test, clf.predict_proba(X_test_complete)[:, 1]))

0.7664713414174888


Можно заключить, что рассматриваемые данные имеют скрытый потенциал и использование более сложных моделей является тому подтверждением, однако, ограниченный объем данных и сложность рассмотренных моделей не позволяют воспроизвести результаты оригинальной статьи, в которой обучение глубокой нейронной сети на всем наборе данных (11 млн строк) с использованием только Low-level признаков позволяет получить более качественные результаты, чем при использовании High-level признаков.